In [1]:
import pandas as pd
import numpy as np
import requests
from pathlib import Path
from time import sleep

In [ ]:
BASE_URL = "https://data.ademe.fr/data-fair/api/v1/datasets/dpe03existant/lines"
PAGE_SIZE = 10000
TARGET_ROWS = 50000
RAW_DIR = Path("../data/raw")
RAW_PATH = RAW_DIR / f"dpe_logements_{TARGET_ROWS}.parquet"
FORCE_REFRESH = False  # set True only when we want to re-download from ADEME

QS_FILTER = 'type_batiment:"maison" OR type_batiment:"appartement"'


def fetch_logements(target_rows=TARGET_ROWS, page_size=PAGE_SIZE, qs=QS_FILTER):
    params = {"size": page_size, "qs": qs}
    url = BASE_URL
    pages = []
    fetched = 0
    while url and fetched < target_rows:
        r = requests.get(url, params=params if url == BASE_URL else None, timeout=60)
        r.raise_for_status()
        payload = r.json()
        results = payload.get("results", [])
        if not results:
            break
        pages.append(pd.DataFrame(results))
        fetched += len(results)
        print(f"fetched {fetched:>7} / {target_rows} (total available: {payload.get('total')})")
        url = payload.get("next")
        sleep(0.2)
    return pd.concat(pages, ignore_index=True).head(target_rows)


if RAW_PATH.exists() and not FORCE_REFRESH:
    df = pd.read_parquet(RAW_PATH)
    print(f"loaded cached raw data from {RAW_PATH} ({RAW_PATH.stat().st_size / 1e6:.1f} MB)")
else:
    RAW_DIR.mkdir(parents=True, exist_ok=True)
    df = fetch_logements()
    df.to_parquet(RAW_PATH, index=False)
    print(f"saved raw data to {RAW_PATH} ({RAW_PATH.stat().st_size / 1e6:.1f} MB)")

df.shape


In [ ]:
print("rows:", len(df))
print("cols:", df.shape[1])
print("\ntype_batiment counts:")
print(df["type_batiment"].value_counts(dropna=False))

rows: 50000
cols: 220

type_batiment counts:
type_batiment
appartement    33544
maison         16456
Name: count, dtype: int64


In [ ]:
# Optional explicit save cell. The fetch/load cell above already caches the raw extract.
RAW_DIR.mkdir(parents=True, exist_ok=True)
df.to_parquet(RAW_PATH, index=False)
print(f"saved {RAW_PATH} ({RAW_PATH.stat().st_size / 1e6:.1f} MB)")


In [ ]:
raw_path = globals().get("RAW_PATH", Path("../data/raw/dpe_logements_50000.parquet"))
if "df" not in globals():
    df = pd.read_parquet(raw_path)
    print(f"loaded cached raw data from {raw_path}: {df.shape}")

TARGET_YEAR = "annee_construction"
TARGET_ERA = "construction_era"
TARGET_BINARY = "pre_post_1975"
LEAKY_PERIOD_COLUMN = "periode_construction"  

# Energy features
ENERGY_FEATURES = [
    "conso_5_usages_par_m2_ep",        # kWh/m2/year - standardised total
    "emission_ges_5_usages_par_m2",    # kg CO2/m2/year
    "etiquette_dpe",                   # A-G label (ordinal)
    "etiquette_ges",                   # A-G label (ordinal)
    "conso_chauffage_ep",              # kWh/m2/year heating
    "conso_ecs_ep",                    # kWh/m2/year hot water
    "conso_refroidissement_ep",        # kWh/m2/year cooling
    "conso_eclairage_ep",              # kWh/m2/year lighting (post-2021 only)
    "conso_auxiliaires_ep",            # kWh/m2/year auxiliaries (post-2021 only)
]

# Structural features
STRUCTURAL_FEATURES = [
    "surface_habitable_logement",      # m2 habitable surface
    "type_batiment",                   # maison vs. appartement
    "nombre_niveau_logement",          # number of floors
    "qualite_isolation_murs",          # wall insulation type/quality
    "qualite_isolation_menuiseries",   # window glazing type
    "type_generateur_chauffage_principal",  # heating system type
    "type_energie_principale_chauffage",    # heating energy
    "type_installation_ecs",           # hot water system type
    "type_energie_principale_ecs",     # hot water energy type
    "type_ventilation",                # natural vs. mechanical ventilation
    "isolation_toiture",               # roof insulation type/quality
    "qualite_isolation_plancher_bas",  # floor insulation type/quality
    "classe_inertie_batiment",         # thermal mass / inertia class
]

# Geographic features
GEO_FEATURES = [
    "code_departement_ban",   # French department code
    "zone_climatique",        # climate zone (H1, H2, H3)
    "classe_altitude",        # altitude category
]

ALL_FEATURES = ENERGY_FEATURES + STRUCTURAL_FEATURES + GEO_FEATURES
REQUIRED_COLUMNS = [TARGET_YEAR] + ALL_FEATURES

print(f"Target source: {TARGET_YEAR} -> {TARGET_ERA} and {TARGET_BINARY}")
print(f"Features to keep: {len(ALL_FEATURES)} ({len(ENERGY_FEATURES)} energy, {len(STRUCTURAL_FEATURES)} structural, {len(GEO_FEATURES)} geo)")

missing = [c for c in REQUIRED_COLUMNS if c not in df.columns]
if missing:
    raise KeyError(f"Missing required columns: {missing}")
print("All required columns present in data")

# Keep the exact construction year only for target creation. Do not include period/year in model features.
df_selected = df[REQUIRED_COLUMNS].copy()
print(f"\nShape after feature selection: {df_selected.shape}")
print(f"Removed {df.shape[1] - df_selected.shape[1]} administrative/ID/address/leaky columns")


### Cleaning pipeline

This section implements :
- drop rows with missing/invalid construction year;
- drop implausible surfaces (`< 10` or `> 500` m2);
- drop implausible floor counts (`< 1` or `> 20`);
- bin construction year into the six era classes;
- create the binary pre/post-1975 target from the revised spec;
- impute remaining missing values with median/mode;
- produce ordinal-encoded data for tree models and one-hot encoded data for linear models;
- log the fraction of rows removed at each cleaning step.


In [ ]:
from datetime import date

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler

SURFACE_COL = "surface_habitable_logement"
FLOORS_COL = "nombre_niveau_logement"
MIN_VALID_YEAR = 1000  
MAX_VALID_YEAR = date.today().year

ERA_LABELS = [
    "pre-1948",
    "1948-1974",
    "1975-1988",
    "1989-2000",
    "2001-2012",
    "2013+",
]
ERA_BINS = [-np.inf, 1947, 1974, 1988, 2000, 2012, np.inf]
BINARY_THRESHOLD_YEAR = 1975
BINARY_LABELS = ["pre-1975", "post-1975"]

cleaning_log = []
df_work = df_selected.copy()
start_rows = len(df_work)


def drop_and_log(data, mask_to_keep, step, reason):
    """Drop rows and record both step-level and original-dataset fractions."""
    before = len(data)
    cleaned = data.loc[mask_to_keep].copy()
    dropped = before - len(cleaned)
    cleaning_log.append(
        {
            "step": step,
            "reason": reason,
            "rows_before": before,
            "rows_dropped": dropped,
            "rows_after": len(cleaned),
            "dropped_fraction_of_previous": dropped / before if before else 0,
            "dropped_fraction_of_start": dropped / start_rows if start_rows else 0,
        }
    )
    return cleaned

# Coerce fields used in validity checks to numeric before filtering.
df_work[TARGET_YEAR] = pd.to_numeric(df_work[TARGET_YEAR], errors="coerce")
df_work[SURFACE_COL] = pd.to_numeric(df_work[SURFACE_COL], errors="coerce")
df_work[FLOORS_COL] = pd.to_numeric(df_work[FLOORS_COL], errors="coerce")

year_valid = df_work[TARGET_YEAR].between(MIN_VALID_YEAR, MAX_VALID_YEAR, inclusive="both")
df_work = drop_and_log(
    df_work,
    year_valid,
    "construction_year",
    f"missing/non-numeric year or outside [{MIN_VALID_YEAR}, {MAX_VALID_YEAR}]",
)

surface_valid = df_work[SURFACE_COL].between(10, 500, inclusive="both")
df_work = drop_and_log(
    df_work,
    surface_valid,
    "surface",
    "surface_habitable_logement outside [10, 500] m2",
)

floors_valid = df_work[FLOORS_COL].between(1, 20, inclusive="both")
df_work = drop_and_log(
    df_work,
    floors_valid,
    "floors",
    "nombre_niveau_logement outside [1, 20]",
)

# Primary classification target: regulation-aligned era bins from project_spec.pdf section 5.
df_work[TARGET_ERA] = pd.cut(
    df_work[TARGET_YEAR],
    bins=ERA_BINS,
    labels=ERA_LABELS,
    right=True,
    ordered=True,
)

era_valid = df_work[TARGET_ERA].notna()
df_work = drop_and_log(
    df_work,
    era_valid,
    "era_binning",
    "construction year could not be assigned to one of the six era bins",
)

# Revised spec binary target: pre/post the RT1974 regulatory break, represented as pre/post-1975.
df_work[TARGET_BINARY] = np.where(
    df_work[TARGET_YEAR] >= BINARY_THRESHOLD_YEAR,
    BINARY_LABELS[1],
    BINARY_LABELS[0],
)

cleaning_report = pd.DataFrame(cleaning_log)
cleaning_report["dropped_pct_of_previous"] = (100 * cleaning_report["dropped_fraction_of_previous"]).round(2)
cleaning_report["dropped_pct_of_start"] = (100 * cleaning_report["dropped_fraction_of_start"]).round(2)

print(f"Rows before cleaning: {start_rows:,}")
print(f"Rows after cleaning:  {len(df_work):,}")
print(f"Total dropped:        {start_rows - len(df_work):,} ({(1 - len(df_work) / start_rows) * 100:.2f}%)")
cleaning_report[
    [
        "step",
        "reason",
        "rows_before",
        "rows_dropped",
        "rows_after",
        "dropped_pct_of_previous",
        "dropped_pct_of_start",
    ]
]


In [ ]:
# Sanity checks 
era_distribution = (
    df_work[TARGET_ERA]
    .value_counts(sort=False)
    .rename_axis(TARGET_ERA)
    .reset_index(name="rows")
)
era_distribution["pct"] = (100 * era_distribution["rows"] / len(df_work)).round(2)
binary_distribution = (
    df_work[TARGET_BINARY]
    .value_counts(sort=False)
    .reindex(BINARY_LABELS)
    .rename_axis(TARGET_BINARY)
    .reset_index(name="rows")
)
binary_distribution["pct"] = (100 * binary_distribution["rows"] / len(df_work)).round(2)

year_summary = df_work[TARGET_YEAR].describe(percentiles=[0.01, 0.25, 0.5, 0.75, 0.99])

print("Construction year summary after filtering:")
print(year_summary)
print("\nEra class balance:")
print(era_distribution.to_string(index=False))
print("\nBinary target balance:")
print(binary_distribution.to_string(index=False))


### Imputation and encoding

The feature matrix excludes both `annee_construction` and ADEME's `periode_construction`, because both directly reveal the target. Numeric features use median imputation. Categorical features use mode imputation. The tree matrix uses ordinal categorical codes; the linear matrix uses one-hot categorical columns and scaled numeric columns.


In [ ]:
X_raw = df_work[ALL_FEATURES].copy()
y_era = df_work[TARGET_ERA].astype(str).copy()
y_binary = df_work[TARGET_BINARY].astype(str).copy()
y_year = df_work[TARGET_YEAR].copy()  # optional regression target 

numeric_features = X_raw.select_dtypes(include=["number"]).columns.tolist()
categorical_features = [c for c in ALL_FEATURES if c not in numeric_features]

# Normalize missing categorical markers so SimpleImputer treats None/NA values consistently.
if categorical_features:
    X_raw[categorical_features] = X_raw[categorical_features].where(
        X_raw[categorical_features].notna(),
        np.nan,
    )

try:
    one_hot_encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
except TypeError:  # compatibility with older scikit-learn
    one_hot_encoder = OneHotEncoder(handle_unknown="ignore", sparse=False)

tree_preprocessor = ColumnTransformer(
    transformers=[
        ("num", SimpleImputer(strategy="median"), numeric_features),
        (
            "cat",
            Pipeline(
                steps=[
                    ("imputer", SimpleImputer(strategy="most_frequent")),
                    ("encoder", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)),
                ]
            ),
            categorical_features,
        ),
    ],
    verbose_feature_names_out=False,
)

# Also keep a readable imputed version before categorical encoding for EDA/handoff.
X_imputed = X_raw.copy()
if numeric_features:
    numeric_imputer = SimpleImputer(strategy="median")
    X_imputed[numeric_features] = numeric_imputer.fit_transform(X_raw[numeric_features])
if categorical_features:
    categorical_imputer = SimpleImputer(strategy="most_frequent")
    X_imputed[categorical_features] = categorical_imputer.fit_transform(X_raw[categorical_features])

linear_preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            Pipeline(
                steps=[
                    ("imputer", SimpleImputer(strategy="median")),
                    ("scaler", StandardScaler()),
                ]
            ),
            numeric_features,
        ),
        (
            "cat",
            Pipeline(
                steps=[
                    ("imputer", SimpleImputer(strategy="most_frequent")),
                    ("encoder", one_hot_encoder),
                ]
            ),
            categorical_features,
        ),
    ]
)

X_tree_array = tree_preprocessor.fit_transform(X_raw)
X_tree = pd.DataFrame(
    X_tree_array,
    columns=tree_preprocessor.get_feature_names_out(),
    index=X_raw.index,
)

X_linear_array = linear_preprocessor.fit_transform(X_raw)
X_linear = pd.DataFrame(
    X_linear_array,
    columns=linear_preprocessor.get_feature_names_out(),
    index=X_raw.index,
)

print(f"Raw feature matrix:       {X_raw.shape}")
print(f"Tree ordinal matrix:      {X_tree.shape}")
print(f"Linear one-hot matrix:    {X_linear.shape}")
print(f"Era target rows:          {y_era.shape[0]}")
print(f"Binary target rows:       {y_binary.shape[0]}")
print(f"Numeric features:         {len(numeric_features)}")
print(f"Categorical features:     {len(categorical_features)}")
print(f"Missing values in X_imputed: {int(X_imputed.isna().sum().sum())}")
print(f"Missing values in X_tree: {int(X_tree.isna().sum().sum())}")
print(f"Missing values in X_linear: {int(X_linear.isna().sum().sum())}")


In [ ]:
# Save for the modeling notebook
clean_dir = Path("../data/cleaned")
clean_dir.mkdir(parents=True, exist_ok=True)

model_ready = df_work[[TARGET_YEAR, TARGET_ERA, TARGET_BINARY] + ALL_FEATURES].copy()
model_ready_path = clean_dir / "dpe_clean_model_ready.parquet"
imputed_path = clean_dir / "dpe_clean_imputed.parquet"
tree_path = clean_dir / "dpe_tree_ordinal.parquet"
linear_path = clean_dir / "dpe_linear_onehot.parquet"
target_path = clean_dir / "dpe_targets.parquet"
log_path = clean_dir / "cleaning_log.csv"

model_ready.to_parquet(model_ready_path, index=False)
targets = {TARGET_ERA: y_era.values, TARGET_BINARY: y_binary.values, TARGET_YEAR: y_year.values}
X_imputed.assign(**targets).to_parquet(imputed_path, index=False)
X_tree.assign(**targets).to_parquet(tree_path, index=False)
X_linear.assign(**targets).to_parquet(linear_path, index=False)
pd.DataFrame(targets).to_parquet(target_path, index=False)
cleaning_report.to_csv(log_path, index=False)

print("Saved cleaned deliverables:")
for p in [model_ready_path, imputed_path, tree_path, linear_path, target_path, log_path]:
    print(f"- {p} ({p.stat().st_size / 1e6:.2f} MB)")
